In [14]:
import pandas as pd
import numpy as np
from pathlib import Path

# Topography per Italian region
- Clip SRTM 30 m DEM to Italy with limits_IT_regions.geojson
- Compute elevation spread (mean/median/std/range/IQR) and slope-based ruggedness per region
- Export metrics to CSV plus an enriched GeoJSON

In [15]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
from rasterstats import zonal_stats
import rasterio
from rasterio.merge import merge
import requests
import gzip
import shutil

data_dir = Path("c:/Users/juanx/Documents/GitHub/juanxgi83.github.io/data")
regions_path = data_dir / "limits_IT_regions.geojson"
work_dir = Path("C:/Users/juanx/Documents/PP434/topography_cache")
tile_dir = work_dir / "srtm_tiles"
work_dir.mkdir(parents=True, exist_ok=True)
tile_dir.mkdir(parents=True, exist_ok=True)

gdf = gpd.read_file(regions_path).to_crs("EPSG:4326")
print("Region columns:", [c for c in gdf.columns if c != gdf.geometry.name])

def tile_names(bounds):
    minx, miny, maxx, maxy = bounds
    lons = range(int(np.floor(minx)), int(np.ceil(maxx)))
    lats = range(int(np.floor(miny)), int(np.ceil(maxy)))
    names = []
    for lat in lats:
        for lon in lons:
            ns = "N" if lat >= 0 else "S"
            ew = "E" if lon >= 0 else "W"
            names.append(f"{ns}{abs(lat):02d}{ew}{abs(lon):03d}")
    return names

def download_and_extract(tile):
    folder = tile[:3]
    url = f"https://elevation-tiles-prod.s3.amazonaws.com/skadi/{folder}/{tile}.hgt.gz"
    gz_path = tile_dir / f"{tile}.hgt.gz"
    hgt_path = tile_dir / f"{tile}.hgt"
    if hgt_path.exists():
        return hgt_path
    if not gz_path.exists():
        print("Downloading", tile)
        resp = requests.get(url, timeout=60)
        resp.raise_for_status()
        gz_path.write_bytes(resp.content)
    with gzip.open(gz_path, "rb") as f_in, open(hgt_path, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    return hgt_path

bounds = gdf.total_bounds
tiles = tile_names(bounds)
print(f"Need {len(tiles)} tiles; downloading/mosaicking...")
tile_paths = [download_and_extract(t) for t in tiles]

srcs = [rasterio.open(p) for p in tile_paths]
mosaic, out_transform = merge(srcs)
for src in srcs:
    src.close()

profile = srcs[0].profile
profile.update({"height": mosaic.shape[1], "width": mosaic.shape[2], "transform": out_transform, "driver": "GTiff", "nodata": -32768})
dem_path = work_dir / "italy_srtm1.tif"
with rasterio.open(dem_path, "w", **profile) as dst:
    dst.write(mosaic)

print("DEM saved to", dem_path)
dem_path

Region columns: ['reg_name', 'reg_istat_code_num', 'reg_istat_code']
Need 169 tiles; downloading/mosaicking...
DEM saved to C:\Users\juanx\Documents\PP434\topography_cache\italy_srtm1.tif


WindowsPath('C:/Users/juanx/Documents/PP434/topography_cache/italy_srtm1.tif')

In [16]:
nodata_val = -9999.0
slope_path = work_dir / "italy_slope_deg.tif"

with rasterio.open(dem_path) as src:
    dem = src.read(1, masked=True).astype("float32")
    dem = dem.filled(np.nan)
    transform = src.transform
    xres_deg = transform.a
    yres_deg = -transform.e
    # convert degree spacing to meters at mid-latitude of Italy
    lat0 = float((bounds[1] + bounds[3]) / 2)
    m_per_deg_lat = 111320.0
    m_per_deg_lon = 111320.0 * np.cos(np.deg2rad(lat0))
    xres_m = xres_deg * m_per_deg_lon
    yres_m = yres_deg * m_per_deg_lat
    gy, gx = np.gradient(dem, yres_m, xres_m)
    slope_rad = np.arctan(np.sqrt(gx**2 + gy**2))
    slope_deg = np.degrees(slope_rad)
    profile = src.profile
    profile.update(dtype="float32", nodata=nodata_val)
    slope_clean = np.where(np.isfinite(slope_deg), slope_deg, nodata_val).astype("float32")
    with rasterio.open(slope_path, "w", **profile) as dst:
        dst.write(slope_clean, 1)

def _finite(arr):
    arr = np.asarray(arr)
    arr = arr[np.isfinite(arr)]
    return arr

def p25(arr):
    arr = _finite(arr)
    return np.percentile(arr, 25) if arr.size else np.nan

def p75(arr):
    arr = _finite(arr)
    return np.percentile(arr, 75) if arr.size else np.nan

def p95(arr):
    arr = _finite(arr)
    return np.percentile(arr, 95) if arr.size else np.nan

elev_stats = zonal_stats(
    gdf,
    dem_path,
    stats=["mean", "median", "std", "min", "max"],
    nodata=nodata_val,
    add_stats={"p25": p25, "p75": p75}
)

slope_stats = zonal_stats(
    gdf,
    slope_path,
    stats=["mean", "std", "max"],
    nodata=nodata_val,
    add_stats={"p95": p95}
)

elev_df = pd.DataFrame(elev_stats).add_prefix("elev_")
slope_df = pd.DataFrame(slope_stats).add_prefix("slope_")
result = pd.concat([gdf.reset_index(drop=True), elev_df, slope_df], axis=1)
result["elev_range"] = result["elev_max"] - result["elev_min"]
result["elev_iqr"] = result["elev_p75"] - result["elev_p25"]
result["rugged_score"] = result["slope_mean"] + result["slope_std"] + (result["elev_range"] / 100.0)  # simple composite
result["slope_p95"] = result["slope_p95"]

region_col_candidates = [c for c in gdf.columns if c != gdf.geometry.name and c.lower().startswith(("name", "reg"))]
region_col = region_col_candidates[0] if region_col_candidates else [c for c in gdf.columns if c != gdf.geometry.name][0]

out_csv_cache = work_dir / "italy_region_topography_metrics.csv"
out_csv_data = data_dir / "italy_region_topography_metrics.csv"
summary_cols = [region_col, "elev_mean", "elev_median", "elev_std", "elev_range", "elev_iqr", "elev_p25", "elev_p75", "slope_mean", "slope_std", "slope_max", "slope_p95", "rugged_score"]
result[summary_cols].to_csv(out_csv_cache, index=False)
result[summary_cols].to_csv(out_csv_data, index=False)

out_geojson_cache = work_dir / "limits_IT_regions_with_topography.geojson"
out_geojson_data = data_dir / "limits_IT_regions_with_topography.geojson"
result.to_file(out_geojson_cache, driver="GeoJSON")
result.to_file(out_geojson_data, driver="GeoJSON")

print("Saved:", out_csv_cache)
print("Saved copy to data/:", out_csv_data)
print("Saved:", out_geojson_cache)
print("Saved copy to data/:", out_geojson_data)
result[[region_col, "rugged_score", "elev_range", "slope_mean", "slope_std", "slope_p95"]].head()

Saved: C:\Users\juanx\Documents\PP434\topography_cache\italy_region_topography_metrics.csv
Saved copy to data/: c:\Users\juanx\Documents\GitHub\juanxgi83.github.io\data\italy_region_topography_metrics.csv
Saved: C:\Users\juanx\Documents\PP434\topography_cache\limits_IT_regions_with_topography.geojson
Saved copy to data/: c:\Users\juanx\Documents\GitHub\juanxgi83.github.io\data\limits_IT_regions_with_topography.geojson


,reg_name,rugged_score,elev_range,slope_mean,slope_std,slope_p95
0,Piemonte,73.142579,4517.0,14.607297,13.365282,42.192936
1,Valle d'Aosta/Vallée d'Aoste,84.637845,4500.0,27.074195,12.563651,49.122356
2,Lombardia,65.295687,3964.0,11.961573,13.694114,40.074085
3,Trentino-Alto Adige/Südtirol,74.785425,3819.0,24.157186,12.438239,45.488281
4,Veneto,54.615860,3275.0,9.295405,12.570455,39.203739


# Weather factors per region (Open-Meteo ERA5, 2024)
- Fetch daily ERA5 reanalysis for each region centroid via Open-Meteo
- Aggregate yearly precipitation, snow, temperature, wind, sunshine, and cloudiness
- Join with topography metrics for a combined region-level feature table

In [17]:
import json
import time
import random
import requests

# Build a tidy dataframe of the topography metrics already computed
summary_df = result[summary_cols].copy()

# Use a projected CRS for centroids to avoid geographic distortions, then convert back to WGS84
proj_crs = "EPSG:32632"  # UTM zone covering most of Italy
wgs84 = "EPSG:4326"
gdf_proj = gdf.to_crs(proj_crs)
centroids_proj = gdf_proj.geometry.centroid
centroids = gpd.GeoDataFrame(gdf_proj[[region_col]].copy(), geometry=centroids_proj, crs=proj_crs).to_crs(wgs84)
centroids["centroid_lon"] = centroids.geometry.x
centroids["centroid_lat"] = centroids.geometry.y

openmeteo_url = "https://archive-api.open-meteo.com/v1/era5"
daily_vars = [
    "precipitation_sum",        # mm of liquid + snow melt
    "rain_sum",                 # mm rain only
    "snowfall_sum",             # cm snow
    "precipitation_hours",      # hours with precipitation > 0.1 mm
    "temperature_2m_mean",
    "temperature_2m_min",
    "temperature_2m_max",
    "wind_speed_10m_max",       # m/s
    "wind_gusts_10m_max",       # m/s
    "shortwave_radiation_sum",  # MJ/m^2
    "sunshine_duration",        # seconds
    "cloud_cover_mean"          # %
]

agg_map = {
    "precipitation_sum": "sum",
    "rain_sum": "sum",
    "snowfall_sum": "sum",
    "precipitation_hours": "sum",
    "temperature_2m_mean": "mean",
    "temperature_2m_min": "min",
    "temperature_2m_max": "max",
    "wind_speed_10m_max": "max",
    "wind_gusts_10m_max": "max",
    "shortwave_radiation_sum": "sum",
    "sunshine_duration": "sum",
    "cloud_cover_mean": "mean"
}

# Rename weather fields with units for clarity
rename_map = {
    "precipitation_sum": "precip_sum_mm",
    "rain_sum": "rain_sum_mm",
    "snowfall_sum": "snowfall_sum_cm",
    "precipitation_hours": "precip_hours_hr",
    "temperature_2m_mean": "temp_mean_c",
    "temperature_2m_min": "temp_min_c",
    "temperature_2m_max": "temp_max_c",
    "wind_speed_10m_max": "wind_speed10m_max_ms",
    "wind_gusts_10m_max": "wind_gust10m_max_ms",
    "shortwave_radiation_sum": "shortwave_rad_sum_MJm2",
    "sunshine_duration": "sunshine_duration_s",
    "cloud_cover_mean": "cloud_cover_mean_pct"
}

session = requests.Session()
session.headers.update({"User-Agent": "italy-accidents-weather/1.0 (https://github.com/juanxgi83)"})
initial_pause_sec = 8.0  # pause before first call to avoid bursts
base_delay = 12.0        # base backoff for transient errors
region_pause_sec = 90.0  # pause between regions to be extra polite
retry_429_floor = 180.0  # minimum wait after a 429

# Limit regions while testing; set to None for full run
region_limit = None
centroids_to_fetch = centroids if region_limit is None else centroids.head(region_limit)


def _sleep_with_jitter(seconds: float):
    time.sleep(seconds * (1 + random.random() * 0.3))


def fetch_weather_year(lat: float, lon: float, start_date: str = "2024-01-01", end_date: str = "2024-12-31", max_retries: int = 10) -> pd.Series:
    """Fetch and aggregate daily ERA5 stats for one point with retry/backoff."""
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "daily": ",".join(daily_vars),
        "timezone": "Europe/Rome"
    }
    for attempt in range(max_retries):
        try:
            resp = session.get(openmeteo_url, params=params, timeout=90)
            resp.raise_for_status()
            payload = resp.json()
            if "daily" not in payload:
                raise ValueError("Open-Meteo response missing 'daily'")
            daily_df = pd.DataFrame(payload["daily"])
            daily_df["time"] = pd.to_datetime(daily_df["time"])
            summary = daily_df.agg(agg_map)
            summary["latitude"] = payload.get("latitude", lat)
            summary["longitude"] = payload.get("longitude", lon)
            return summary
        except (requests.HTTPError, requests.ConnectionError, requests.Timeout) as exc:
            status = exc.response.status_code if hasattr(exc, "response") and exc.response is not None else None
            if attempt < max_retries - 1 and (status in (429, 502, 503, 504) or status is None):
                if status == 429:
                    wait_sec = max(retry_429_floor, base_delay * (attempt + 1) * 1.8)
                else:
                    wait_sec = base_delay * (attempt + 1) * 1.5
                print(f"Retry {attempt+1}/{max_retries} for {lat:.3f},{lon:.3f}; status {status}; sleeping {wait_sec:.1f}s")
                _sleep_with_jitter(wait_sec)
                continue
            raise


print(f"Starting fetch for {len(centroids_to_fetch)} regions; initial pause {initial_pause_sec}s...")
_sleep_with_jitter(initial_pause_sec)

weather_records = []
for _, row in centroids_to_fetch.iterrows():
    stats = fetch_weather_year(row["centroid_lat"], row["centroid_lon"])
    stats[region_col] = row[region_col]
    weather_records.append(stats)
    _sleep_with_jitter(region_pause_sec)  # generous pause between regions

weather_df = pd.DataFrame(weather_records).rename(columns=rename_map)

# Merge topo + weather for a single feature table
merged_df = summary_df.merge(weather_df, on=region_col)

# Save copies alongside other outputs
weather_out_cache = work_dir / "italy_region_topography_weather_2024.csv"
weather_out_data = data_dir / "italy_region_topography_weather_2024.csv"
merged_df.to_csv(weather_out_cache, index=False)
merged_df.to_csv(weather_out_data, index=False)

print("Saved:", weather_out_cache)
print("Saved copy to data/:", weather_out_data)
print("Preview rows:")
merged_df.head()

Starting fetch for 20 regions; initial pause 8.0s...
Saved: C:\Users\juanx\Documents\PP434\topography_cache\italy_region_topography_weather_2024.csv
Saved copy to data/: c:\Users\juanx\Documents\GitHub\juanxgi83.github.io\data\italy_region_topography_weather_2024.csv
Preview rows:


,reg_name,elev_mean,elev_median,elev_std,elev_range,elev_iqr,elev_p25,elev_p75,slope_mean,slope_std,...,temp_mean_c,temp_min_c,temp_max_c,wind_speed10m_max_ms,wind_gust10m_max_ms,shortwave_rad_sum_MJm2,sunshine_duration_s,cloud_cover_mean_pct,latitude,longitude
0,Piemonte,805.030570,431.0,753.768992,4517.0,1632.0,251.0,1883.0,14.607297,13.365282,...,13.080601,-5.6,33.0,28.2,53.3,4850.05,10570150.38,61.374317,45.026360,7.965838
1,Valle d'Aosta/Vallée d'Aoste,2103.310008,2179.0,709.354148,4500.0,996.0,1675.0,2671.0,27.074195,12.563651,...,13.494809,-8.5,35.2,15.1,95.0,5286.05,11257343.90,61.937158,45.729347,7.381703
2,Lombardia,615.719900,206.0,771.706171,3964.0,1234.0,83.0,1317.0,11.961573,13.694114,...,13.730328,-4.9,33.5,37.5,68.0,4836.67,10890507.01,58.133880,45.588750,9.764151
3,Trentino-Alto Adige/Südtirol,1593.798690,1607.0,680.294140,3819.0,1195.0,912.0,2107.0,24.157186,12.438239,...,12.411749,-9.7,33.6,14.9,59.0,5066.05,11521484.89,59.532787,46.432335,11.250000
4,Veneto,400.689873,41.0,624.253663,3275.0,1121.0,4.0,1125.0,9.295405,12.570455,...,14.470765,-4.8,35.5,38.7,72.0,4936.10,11386790.84,59.833333,45.659050,11.905513


In [18]:
# Add 2024 per-capita vehicle metrics from master file and merge into merged_df
weather_csv_path = data_dir / "italy_region_topography_weather_2024.csv"
merged_df = pd.read_csv(weather_csv_path)

master_path = data_dir / "italy_master_file_generated_from_regression_ipynb.json"
master_df = pd.read_json(master_path)

vehicles_cols = [c for c in master_df.columns if "per capita (vehicles" in c.lower() or "per capita (fleet" in c.lower()]
veh_2024 = master_df.loc[master_df["TIME_PERIOD"] == 2024, ["Territory"] + vehicles_cols].copy()

def _norm_region(s: pd.Series) -> pd.Series:
    s = s.str.replace(r"\s*/\s*", " / ", regex=True)
    s = s.str.replace(r"Trentino[- ]Alto Adige\s*/\s*S.*tirol", "Trentino Alto Adige / Südtirol", regex=True, case=False)
    s = s.str.replace(r"Valle d['’]Aosta.*", "Valle d'Aosta / Vallée d'Aoste", regex=True, case=False)
    return s

merged_df[region_col] = _norm_region(merged_df[region_col])
veh_2024["Territory"] = _norm_region(veh_2024["Territory"])
veh_2024 = veh_2024.rename(columns={"Territory": region_col})

# Drop any vehicle columns already present (_x/_y or otherwise) before merging fresh values
existing_vehicle_cols = [c for c in merged_df.columns if ("per capita (vehicles" in c.lower()) or ("per capita (fleet" in c.lower())]
merged_df = merged_df.drop(columns=existing_vehicle_cols, errors="ignore")
merged_df = merged_df.merge(veh_2024, on=region_col, how="left")

# Persist the enriched table so the next cell (fixed effects) can reuse it
merged_df.to_csv(weather_csv_path, index=False)
merged_df.to_csv(data_dir / "italy_region_topography_weather_2024.csv", index=False)
merged_df.head()

,reg_name,elev_mean,elev_median,elev_std,elev_range,elev_iqr,elev_p25,elev_p75,slope_mean,slope_std,...,longitude,Buses or trolley-buses per capita (vehicles),Motor cars per capita (vehicles),Motor units per capita (vehicles),Motorcycles per capita (vehicles),Other vehicles per capita (vehicles),Three-wheelers or motor-vans per capita (vehicles),Total vehicles per capita (fleet),Trailers per capita (vehicles),Trucks per capita (vehicles)
0,Piemonte,805.030570,431.0,753.768992,4517.0,1632.0,251.0,1883.0,14.607297,13.365282,...,7.965838,0.001285,0.723291,0.003382,0.122746,2.352000e-07,0.004887,0.964694,0.006940,0.102162
1,Valle d'Aosta / Vallée d'Aoste,2103.310008,2179.0,709.354148,4500.0,996.0,1675.0,2671.0,27.074195,12.563651,...,7.381703,0.002092,1.930076,0.002092,0.147774,0.000000e+00,0.029265,2.634985,0.005151,0.518535
2,Lombardia,615.719900,206.0,771.706171,3964.0,1234.0,83.0,1317.0,11.961573,13.694114,...,9.764151,0.001085,0.641859,0.003329,0.123396,1.998000e-07,0.002815,0.855711,0.006875,0.076351
3,Trentino Alto Adige / Südtirol,1593.798690,1607.0,680.294140,3819.0,1195.0,912.0,2107.0,24.157186,12.438239,...,11.250000,0.002318,1.192616,0.008236,0.130484,0.000000e+00,0.007604,1.562714,0.017403,0.204053
4,Veneto,400.689873,41.0,624.253663,3275.0,1121.0,4.0,1125.0,9.295405,12.570455,...,11.905513,0.001412,0.680668,0.004965,0.114765,0.000000e+00,0.002420,0.906350,0.010019,0.092101


In [19]:
# Attach significant region fixed effects (json) as first column after region name
weather_csv_path = data_dir / "italy_region_topography_weather_2024.csv"
merged_df = pd.read_csv(weather_csv_path)
fe_path = data_dir / "italy_accidents_significant_region_fixed_effects.json"
fe_df = pd.read_json(fe_path)
fe_df = fe_df.rename(columns={"Region": region_col, "Fixed_Effect": "region_fe"})

# Normalize slashes in region names to ensure merge matches (e.g., "A/B" -> "A / B")
merged_df[region_col] = merged_df[region_col].str.replace(r"\s*/\s*", " / ", regex=True)
merged_df[region_col] = merged_df[region_col].replace({"Trentino-Alto Adige / Südtirol": "Trentino Alto Adige / Südtirol"})

# Merge and reorder so region_fe follows reg_name/region_col
merged_df = merged_df.merge(fe_df, on=region_col, how="left")
ordered_cols = [region_col, "region_fe"] + [c for c in merged_df.columns if c not in {region_col, "region_fe"}]
merged_df = merged_df[ordered_cols]

merged_df

,reg_name,region_fe,elev_mean,elev_median,elev_std,elev_range,elev_iqr,elev_p25,elev_p75,slope_mean,...,longitude,Buses or trolley-buses per capita (vehicles),Motor cars per capita (vehicles),Motor units per capita (vehicles),Motorcycles per capita (vehicles),Other vehicles per capita (vehicles),Three-wheelers or motor-vans per capita (vehicles),Total vehicles per capita (fleet),Trailers per capita (vehicles),Trucks per capita (vehicles)
0,Piemonte,-300.434783,805.030570,431.0,753.768992,4517.0,1632.0,251.0,1883.0,14.607297,...,7.965838,0.001285,0.723291,0.003382,0.122746,2.352000e-07,0.004887,0.964694,0.006940,0.102162
1,Valle d'Aosta / Vallée d'Aoste,0.000000,2103.310008,2179.0,709.354148,4500.0,996.0,1675.0,2671.0,27.074195,...,7.381703,0.002092,1.930076,0.002092,0.147774,0.000000e+00,0.029265,2.634985,0.005151,0.518535
2,Lombardia,-316.154875,615.719900,206.0,771.706171,3964.0,1234.0,83.0,1317.0,11.961573,...,9.764151,0.001085,0.641859,0.003329,0.123396,1.998000e-07,0.002815,0.855711,0.006875,0.076351
3,Trentino Alto Adige / Südtirol,0.000000,1593.798690,1607.0,680.294140,3819.0,1195.0,912.0,2107.0,24.157186,...,11.250000,0.002318,1.192616,0.008236,0.130484,0.000000e+00,0.007604,1.562714,0.017403,0.204053
4,Veneto,-408.726809,400.689873,41.0,624.253663,3275.0,1121.0,4.0,1125.0,9.295405,...,11.905513,0.001412,0.680668,0.004965,0.114765,0.000000e+00,0.002420,0.906350,0.010019,0.092101
5,Friuli-Venezia Giulia,-247.528687,553.944356,252.0,590.825372,2771.0,1022.0,22.0,1044.0,14.553060,...,13.041401,0.001472,0.693887,0.003469,0.135587,0.000000e+00,0.003903,0.933944,0.007571,0.088056
6,Liguria,507.581486,533.323828,489.0,346.954426,2187.0,1087.0,-468.0,619.0,19.895803,...,8.656488,0.001650,0.563300,0.002270,0.289260,6.626000e-07,0.009497,0.942734,0.004608,0.072150
7,Emilia-Romagna,0.000000,295.580412,108.0,363.644528,2184.0,459.0,1.0,460.0,8.199264,...,11.059908,0.001416,0.692469,0.004189,0.133513,4.492000e-07,0.002927,0.943910,0.008401,0.100994
8,Toscana,0.000000,358.237053,283.0,306.265734,2101.0,456.0,-31.0,425.0,11.548520,...,11.081080,0.001687,0.743856,0.002361,0.165849,0.000000e+00,0.007330,1.028156,0.005418,0.101655
9,Umbria,-259.701394,493.068216,421.0,289.529895,2385.0,405.0,295.0,700.0,12.309172,...,12.436850,0.001709,0.772744,0.005050,0.123676,1.172200e-06,0.008173,1.021501,0.010545,0.099603


In [20]:
import altair as alt

# Scatterplot matrix for region fixed effects vs key predictors
moto_col = "Motorcycles per capita (vehicles)"
numeric_cols = ["region_fe", "rugged_score", "precip_sum_mm", moto_col]
scatter_df = merged_df[[region_col] + numeric_cols].copy()
for col in numeric_cols:
    scatter_df[col] = pd.to_numeric(scatter_df[col], errors="coerce")
scatter_df = scatter_df.dropna(subset=numeric_cols)

# Use safer field names for plotting
plot_df = scatter_df.rename(columns={moto_col: "moto_pc"})

# Field definitions for axes
fields = [
    ("rugged_score", "Rugged score"),
    ("precip_sum_mm", "Precip (mm, 2024)"),
    ("moto_pc", "Motorcycles per capita"),
    ("region_fe", "Region fixed effect")
]

label_map = {
    "rugged_score": "Rugged Score",
    "precip_sum_mm": "Precipitation",
    "moto_pc": "Motorcycles per Capita",
    "region_fe": "Region Fixed Effect"
}

# Shared sizing and spacing
cell_w, cell_h = 140, 140
cell_spacing = 6
base = alt.Chart(plot_df).properties(width=cell_w, height=cell_h)

# Reusable underlined header

def header_chart(label: str):
    text = (
        alt.Chart(pd.DataFrame({"label": [label]}))
        .mark_text(fontWeight="bold", fontSize=14)
        .encode(text="label")
        .properties(width=cell_w, height=18)
    )
    line = (
        alt.Chart(pd.DataFrame({"y": [0]}))
        .mark_rule(size=2)
        .encode(y=alt.Y("y:Q", scale=alt.Scale(domain=[0, 1]), axis=None))
        .properties(width=cell_w, height=4)
    )
    return alt.vconcat(text, line, spacing=0)

def vertical_header_layer(label: str):
    text = (
        alt.Chart(pd.DataFrame({"label": [label]}))
        .mark_text(fontWeight="bold", fontSize=14, angle=270)
        .encode(text="label", x=alt.value(-65), y=alt.value(cell_h / 2))
    )
    line = (
        alt.Chart(pd.DataFrame({"y1": [0], "y2": [cell_h]}))
        .mark_rule(size=2)
        .encode(x=alt.value(-55), y=alt.value(0), y2=alt.value(cell_h - 4))
    )
    return alt.layer(text, line)

rugged_header = header_chart("Rugged Score")
precip_header = header_chart("Precipitation")
moto_header = header_chart("Motorcycles per Capita")
region_fe_header = header_chart("Region Fixed Effect")
vertical_rugged_layer = vertical_header_layer(label_map["rugged_score"])

# Build matrix cells
charts = []
for row_field, row_title in fields:
    row_charts = []
    for col_field, col_title in fields:
        if row_field == col_field:
            diag_chart = (
                base.mark_bar(color="#4169E1")
                .encode(
                    alt.X(col_field, bin=alt.Bin(maxbins=20), title=col_title),
                    alt.Y("count()", title="Count"),
                    tooltip=[
                        alt.Tooltip("count()", title="Count"),
                        alt.Tooltip(region_col, aggregate="values", title="Regions")
                    ]
                )
            )
            if row_field == fields[0][0]:
                diag_body = alt.layer(diag_chart, vertical_rugged_layer)
                chart = alt.vconcat(rugged_header, diag_body, spacing=2)
            else:
                chart = diag_chart
        else:
            points = (
                base.mark_circle(size=60, opacity=0.7, color="#F5B041")
                .encode(
                    alt.X(col_field, title=col_title),
                    alt.Y(row_field, title=row_title),
                    tooltip=[region_col, "region_fe", "rugged_score", "precip_sum_mm", "moto_pc"]
                )
            )
            trend = (
                base.transform_regression(col_field, row_field)
                .mark_line(color="red", size=2, strokeDash=[6, 3])
                .encode(
                    alt.X(col_field, title=col_title),
                    alt.Y(row_field, title=row_title)
                )
            )
            layered = alt.layer(points, trend)
            if col_field == fields[0][0]:
                label_layer = vertical_header_layer(label_map[row_field])
                chart = alt.layer(layered, label_layer)
            elif row_field == fields[0][0] and col_field == fields[1][0]:
                chart = alt.vconcat(precip_header, layered, spacing=2)
            elif row_field == fields[0][0] and col_field == fields[2][0]:
                chart = alt.vconcat(moto_header, layered, spacing=2)
            elif row_field == fields[0][0] and col_field == fields[3][0]:
                chart = alt.vconcat(region_fe_header, layered, spacing=2)
            else:
                chart = layered
        row_charts.append(chart)
    charts.append(alt.hconcat(*row_charts, spacing=cell_spacing))

grid_body = alt.vconcat(*charts, spacing=cell_spacing).resolve_scale(color="independent")

# Display the grid
grid_body

alt.VConcatChart(...)